In [1]:
import pandas as pd
import numpy as np

# Ajusta la ruta según dónde guardaste el CSV
df = pd.read_csv("artifacts/blocks_features.csv", parse_dates=["start", "end"])

df.shape, df.columns

((5566, 12),
 Index(['start', 'end', 'n', 'duration_min', 'mean_rate', 'max_rate',
        'unique_tags', 'dominant_tag_share', 'unique_msgs',
        'dominant_msg_share', 'flood_candidate', 'flood_severity'],
       dtype='str'))

In [2]:
P99_RATE = 180   # p99 alarms/min
P999_RATE = 345  # p99.9 alarms/min

df["is_severe_rate"] = df["max_rate"] >= P999_RATE
df["is_extreme_rate"] = df["max_rate"] >= P99_RATE

df[["max_rate","is_extreme_rate","is_severe_rate"]].describe()

,max_rate
count,5566.000000
mean,29.088573
std,148.867860
min,1.000000
25%,3.000000
50%,9.000000
75%,21.000000
max,7734.000000


In [3]:
def assign_flood_type(row):
    ut = row["unique_tags"]
    um = row["unique_msgs"]
    dts = row["dominant_tag_share"]
    dms = row["dominant_msg_share"]
    mr = row["max_rate"]
    n = row["n"]

    # 1) Mensaje dominante transversal (muchos tags afectados)
    if (mr >= P99_RATE) and (dms >= 0.70) and (ut >= 20):
        return "MESSAGE_DOMINANT"

    # 2) Oscilación / chatter de un punto (tag domina el bloque)
    if (mr >= P99_RATE) and (dts >= 0.85):
        return "OSCILLATION_SINGLE_POINT"

    # 3) Repetición de pocos tags/mensajes (muy poca diversidad)
    if (mr >= P99_RATE) and (ut <= 10) and (dms >= 0.70):
        return "TAG_DOMINANT_REPETITION"

    # 4) Evento masivo con repetición (evento grande + mensaje dominante medio)
    if (mr >= P99_RATE) and (dms >= 0.50) and (ut >= 10):
        return "MASS_EVENT_WITH_REPETITION"

    # 5) No tipificado / no flood (o flood leve)
    return "OTHER_OR_NO_FLOOD"


df["flood_type"] = df.apply(assign_flood_type, axis=1)

df["flood_type"].value_counts()

flood_type
OTHER_OR_NO_FLOOD             5482
MESSAGE_DOMINANT                26
MASS_EVENT_WITH_REPETITION      25
TAG_DOMINANT_REPETITION         20
OSCILLATION_SINGLE_POINT        13
Name: count, dtype: int64

In [4]:
pd.crosstab(df["flood_type"], df["flood_severity"], margins=True)

flood_severity,medium,none,severe,All
flood_type,,,,
MASS_EVENT_WITH_REPETITION,15,8,2,25
MESSAGE_DOMINANT,22,0,4,26
OSCILLATION_SINGLE_POINT,12,0,1,13
OTHER_OR_NO_FLOOD,17,5465,0,5482
TAG_DOMINANT_REPETITION,14,0,6,20
All,80,5473,13,5566


In [5]:
def top_examples(df, flood_type, k=5):
    sub = df[df["flood_type"] == flood_type].copy()
    return sub.sort_values("max_rate", ascending=False).head(k)[
        ["start","end","n","duration_min","max_rate","unique_tags","dominant_tag_share","unique_msgs","dominant_msg_share","flood_severity","flood_candidate"]
    ]

for t in ["MESSAGE_DOMINANT","OSCILLATION_SINGLE_POINT","TAG_DOMINANT_REPETITION","MASS_EVENT_WITH_REPETITION"]:
    print("\n====", t, "====")
    display(top_examples(df, t, k=5))


==== MESSAGE_DOMINANT ====


,start,end,n,duration_min,max_rate,unique_tags,dominant_tag_share,unique_msgs,dominant_msg_share,flood_severity,flood_candidate
2822,2025-05-03 09:07:08,2025-05-03 09:59:14,12453,52.100000,7734,2615,0.070585,100,0.733317,severe,True
1765,2025-02-02 14:08:37,2025-02-02 19:28:44,44880,320.116667,672,27,0.478810,19,0.957620,severe,True
1457,2025-01-02 14:00:30,2025-01-02 19:14:18,27882,313.800000,456,26,0.490209,18,0.980310,severe,True
4080,2025-09-04 02:54:19,2025-09-04 06:39:38,17175,225.316667,378,20,0.306550,14,0.919301,severe,True
3839,2025-08-08 08:55:12,2025-08-08 12:36:36,7782,221.400000,330,80,0.602544,42,0.839244,medium,True



==== OSCILLATION_SINGLE_POINT ====


,start,end,n,duration_min,max_rate,unique_tags,dominant_tag_share,unique_msgs,dominant_msg_share,flood_severity,flood_candidate
651,2024-06-10 00:00:00,2024-06-10 10:33:08,59259,633.133333,432,77,0.975042,40,0.495267,severe,True
4156,2025-09-06 23:48:11,2025-09-07 20:50:20,61974,1262.150000,342,131,0.927050,49,0.470326,medium,True
652,2024-06-10 10:38:55,2024-06-11 01:05:58,64965,867.050000,315,132,0.908474,40,0.493558,medium,True
1145,2024-10-12 04:53:40,2024-10-12 12:21:32,24600,447.866667,291,110,0.911707,49,0.480976,medium,True
567,2024-05-10 00:00:00,2024-05-11 01:44:22,127794,1544.366667,288,114,0.917649,42,0.466289,medium,True



==== TAG_DOMINANT_REPETITION ====


,start,end,n,duration_min,max_rate,unique_tags,dominant_tag_share,unique_msgs,dominant_msg_share,flood_severity,flood_candidate
4138,2025-09-05 21:01:15,2025-09-05 21:20:53,3369,19.633333,576,6,0.325022,4,0.975067,severe,True
5352,2025-12-05 03:32:04,2025-12-05 05:53:15,17694,141.183333,540,9,0.335198,6,0.725331,severe,True
4477,2025-10-05 21:44:20,2025-10-05 21:54:53,2493,10.550000,486,3,0.333333,1,1.000000,severe,True
4498,2025-10-06 07:00:34,2025-10-06 07:25:59,5493,25.416667,468,6,0.332059,3,0.997815,severe,True
4499,2025-10-06 07:31:42,2025-10-06 08:26:18,7875,54.600000,378,9,0.331048,7,0.996952,severe,True



==== MASS_EVENT_WITH_REPETITION ====


,start,end,n,duration_min,max_rate,unique_tags,dominant_tag_share,unique_msgs,dominant_msg_share,flood_severity,flood_candidate
4379,2025-10-04 00:40:01,2025-10-04 01:59:18,11847,79.283333,735,16,0.302102,10,0.906305,severe,True
3072,2025-06-02 11:16:50,2025-06-03 07:09:34,56340,1192.733333,420,454,0.180777,113,0.653461,severe,True
1636,2025-01-07 09:22:29,2025-01-07 10:41:32,2220,79.050000,342,59,0.167568,26,0.509459,none,False
1646,2025-01-07 22:28:33,2025-01-07 23:04:22,1539,35.816667,288,17,0.259259,12,0.777778,medium,True
1712,2025-01-10 13:55:44,2025-01-10 18:59:34,14886,303.833333,270,105,0.310157,52,0.620314,medium,True


In [6]:
audit = (
    df[df["flood_type"].isin(["MESSAGE_DOMINANT","OSCILLATION_SINGLE_POINT","TAG_DOMINANT_REPETITION","MASS_EVENT_WITH_REPETITION"])]
    .sort_values(["flood_type","flood_severity","max_rate"], ascending=[True, False, False])
    .groupby("flood_type")
    .head(2)
    .reset_index(drop=True)
)

audit[["flood_type","start","end","max_rate","unique_tags","dominant_msg_share","dominant_tag_share","flood_severity"]]

,flood_type,start,end,max_rate,unique_tags,dominant_msg_share,dominant_tag_share,flood_severity
0,MASS_EVENT_WITH_REPETITION,2025-10-04 00:40:01,2025-10-04 01:59:18,735,16,0.906305,0.302102,severe
1,MASS_EVENT_WITH_REPETITION,2025-06-02 11:16:50,2025-06-03 07:09:34,420,454,0.653461,0.180777,severe
2,MESSAGE_DOMINANT,2025-05-03 09:07:08,2025-05-03 09:59:14,7734,2615,0.733317,0.070585,severe
3,MESSAGE_DOMINANT,2025-02-02 14:08:37,2025-02-02 19:28:44,672,27,0.957620,0.478810,severe
4,OSCILLATION_SINGLE_POINT,2024-06-10 00:00:00,2024-06-10 10:33:08,432,77,0.495267,0.975042,severe
5,OSCILLATION_SINGLE_POINT,2025-09-06 23:48:11,2025-09-07 20:50:20,342,131,0.470326,0.927050,medium
6,TAG_DOMINANT_REPETITION,2025-09-05 21:01:15,2025-09-05 21:20:53,576,6,0.975067,0.325022,severe
7,TAG_DOMINANT_REPETITION,2025-12-05 03:32:04,2025-12-05 05:53:15,540,9,0.725331,0.335198,severe
